In [ ]:
from langgraph.graph import StateGraph,START, END
from typing import TypedDict

In [ ]:
class Batsman(TypedDict):
    runs:int
    balls:int
    fours:int
    sixes:int

    sr:float
    bpb:float
    boundary_percentage:float


In [ ]:
#partial updates for parallel workflow
def calculate_sr(state:Batsman):
    sr = (state['runs']/state['balls'])/100
    return {'sr': sr}

In [ ]:
def calculate_bpb(state:Batsman):
    bpb = state['balls']/(state['fours']+state['sixes'])
    return {'bpb': bpb}

In [ ]:
def calculate_boundary_percentage(state:Batsman):
    total_runs = state['runs']
    fours = state['fours']
    sixes = state['sixes']
    boundary_percentage = (((fours * 4) + (sixes * 6)) / total_runs)*100
    return {'boundary_percentage': boundary_percentage}

In [ ]:
def summary(state:Batsman):
    summary = f"Runs: {state['runs']}, Balls: {state['balls']}, Fours: {state['fours']}, Sixes: {state['sixes']}, SR: {state['sr']:.2f}, BPB: {state['bpb']:.2f}, Boundary Percentage: {state['boundary_percentage']:.2f}%"
    return {'summary': summary}

In [ ]:
graph = StateGraph(Batsman)
graph.add_node('calculate_sr',calculate_sr)
graph.add_node('calculate_bpb',calculate_bpb)
graph.add_node('calculate_boundary_percentage',calculate_boundary_percentage)
graph.add_node('summary',summary)

graph.add_edge(START,'calculate_sr')
graph.add_edge(START,'calculate_bpb')
graph.add_edge(START,'calculate_boundary_percentage')
graph.add_edge('calculate_sr','summary')
graph.add_edge('calculate_bpb','summary')
graph.add_edge('calculate_boundary_percentage','summary')
graph.add_edge('summary',END)

workflow = graph.compile()


In [ ]:
initial_state = {'runs':100,'balls':50,'fours':10,'sixes':5}
final_state = workflow.invoke(initial_state)
print(final_state['summary'])